In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df=pd.read_csv('healthcare-dataset-stroke-data.csv')
df

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1
...,...,...,...,...,...,...,...,...,...,...,...,...
5105,18234,Female,80.0,1,0,Yes,Private,Urban,83.75,NaN,never smoked,0
5106,44873,Female,81.0,0,0,Yes,Self-employed,Urban,125.20,40.0,never smoked,0
5107,19723,Female,35.0,0,0,Yes,Self-employed,Rural,82.99,30.6,never smoked,0
5108,37544,Male,51.0,0,0,Yes,Private,Rural,166.29,25.6,formerly smoked,0


In [3]:
df['bmi'] = df['bmi'].fillna(df['bmi'].mean())
print("BMI Nulls after:", df['bmi'].isnull().sum())

BMI Nulls after: 0


In [4]:
le = LabelEncoder()
categorical_columns = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

for col in ['gender', 'ever_married', 'Residence_type']:
    df[col] = le.fit_transform(df[col])


df = pd.get_dummies(df, columns=['work_type','smoking_status'], drop_first=True)

df.shape

(5110, 17)

In [5]:
X = df.drop('stroke', axis=1)
y = df['stroke']

In [7]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)
print("Balanced Training shape:", X.shape)
print("New Balanced Data (X_res):", X_res.shape)

Balanced Training shape: (5110, 16)
New Balanced Data (X_res): (9722, 16)


In [8]:
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, random_state=42)

In [10]:
scaler = StandardScaler()
num_cols = ['age', 'avg_glucose_level', 'bmi']
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])
print("Final Training Shape:", X_train.shape)

Final Training Shape: (7777, 16)


In [11]:
!pip install catboost

In [13]:
from sklearn.metrics import classification_report
from catboost import CatBoostClassifier
model_cat = CatBoostClassifier(iterations=500,learning_rate=0.1,depth=6,random_state=42,verbose=0)
model_cat.fit(X_train, y_train)
pred_cat = model_cat.predict(X_test)
print(classification_report(y_test, pred_cat))
print(confusion_matrix(y_test, pred_cat))

              precision    recall  f1-score   support

           0       0.96      0.96      0.96       975
           1       0.96      0.96      0.96       970

    accuracy                           0.96      1945
   macro avg       0.96      0.96      0.96      1945
weighted avg       0.96      0.96      0.96      1945

[[939  36]
 [ 41 929]]


In [ ]:
paper result: [945  30]
              [ 56 914]

In [14]:
from xgboost import XGBClassifier

model_xgb = XGBClassifier(n_estimators=500, learning_rate=0.1, max_depth=6, random_state=42)
model_xgb.fit(X_train, y_train)
pred_xgb = model_xgb.predict(X_test)
print(classification_report(y_test, pred_xgb))
print(confusion_matrix(y_test, pred_xgb))

              precision    recall  f1-score   support

           0       0.96      0.97      0.96       975
           1       0.97      0.96      0.96       970

    accuracy                           0.96      1945
   macro avg       0.96      0.96      0.96      1945
weighted avg       0.96      0.96      0.96      1945

[[942  33]
 [ 36 934]]


In [ ]:
paper result: [[930, 45],
                [ 51, 919]]

In [15]:
from sklearn.ensemble import RandomForestClassifier
model_rf = RandomForestClassifier(n_estimators=100, random_state=42)
model_rf.fit(X_train, y_train)
pred_rf = model_rf.predict(X_test)
print(classification_report(y_test, pred_rf))
print(confusion_matrix(y_test, pred_rf))

              precision    recall  f1-score   support

           0       0.95      0.94      0.95       975
           1       0.94      0.95      0.95       970

    accuracy                           0.95      1945
   macro avg       0.95      0.95      0.95      1945
weighted avg       0.95      0.95      0.95      1945

[[921  54]
 [ 52 918]]


In [ ]:
paper reslt: [[922  53]
              [ 34 936]]

In [16]:
from sklearn.svm import SVC
model_svm = SVC(kernel='rbf', probability=True, random_state=42)
model_svm.fit(X_train, y_train)
pred_svm = model_svm.predict(X_test)
print(classification_report(y_test, pred_svm))
print(confusion_matrix(y_test, pred_svm))

              precision    recall  f1-score   support

           0       0.51      0.52      0.52       975
           1       0.51      0.50      0.50       970

    accuracy                           0.51      1945
   macro avg       0.51      0.51      0.51      1945
weighted avg       0.51      0.51      0.51      1945

[[509 466]
 [488 482]]


In [17]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
model_svm = SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=42)
model_svm.fit(X_train_scaled, y_train)
pred_svm = model_svm.predict(X_test_scaled)
print(classification_report(y_test, pred_svm))
print(confusion_matrix(y_test, pred_svm))

              precision    recall  f1-score   support

           0       0.90      0.90      0.90       975
           1       0.90      0.90      0.90       970

    accuracy                           0.90      1945
   macro avg       0.90      0.90      0.90      1945
weighted avg       0.90      0.90      0.90      1945

[[877  98]
 [ 94 876]]


In [19]:
from sklearn.linear_model import LogisticRegression
model_lr = LogisticRegression(random_state=42, max_iter=1000)
model_lr.fit(X_train, y_train)
pred_lr = model_lr.predict(X_test)
print(classification_report(y_test, pred_lr))
print(confusion_matrix(y_test, pred_rf))

              precision    recall  f1-score   support

           0       0.87      0.86      0.87       975
           1       0.86      0.87      0.87       970

    accuracy                           0.87      1945
   macro avg       0.87      0.87      0.87      1945
weighted avg       0.87      0.87      0.87      1945

[[921  54]
 [ 52 918]]


C:\Users\abc\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [20]:
model_lr = LogisticRegression(random_state=42, max_iter=1000)
model_lr.fit(X_train_scaled, y_train)
pred_lr = model_lr.predict(X_test_scaled)
print(classification_report(y_test, pred_lr))
print(confusion_matrix(y_test, pred_rf))

              precision    recall  f1-score   support

           0       0.87      0.86      0.87       975
           1       0.86      0.87      0.87       970

    accuracy                           0.87      1945
   macro avg       0.87      0.87      0.87      1945
weighted avg       0.87      0.87      0.87      1945

[[921  54]
 [ 52 918]]


In [ ]:
paper result: [726 249
               162  808]

In [21]:
from sklearn.neighbors import KNeighborsClassifier
model_knn = KNeighborsClassifier(n_neighbors=5)
model_knn.fit(X_train_scaled, y_train)
pred_knn = model_knn.predict(X_test_scaled)
print(classification_report(y_test, pred_knn))
print(confusion_matrix(y_test, pred_knn))

              precision    recall  f1-score   support

           0       0.93      0.89      0.91       975
           1       0.90      0.93      0.91       970

    accuracy                           0.91      1945
   macro avg       0.91      0.91      0.91      1945
weighted avg       0.91      0.91      0.91      1945

[[871 104]
 [ 69 901]]


In [ ]:
https://www.semanticscholar.org/paper/Stroke-Prediction-on-Healthcare-Data-Using-SMOTE-Zamil-Islam/c409ffc0c1d4f6cc639f7ec8cd7e4e6f929d3194?utm_source=direct_link